### Análise Exploratória de Dados: `food_estoque_lojas`
**Visão Geral:** Total de Registros, Colunas, Estrutura`(Schema)`.<br>
**Nulos:** Total de registros nulos por colunas.<br>
**Primaky Key:** Total de PK nulos, PK duplicadas e Registros Duplicados.<br>
**Data Máxima e Mínima:** Range de datas da tabela.<br>
**Valores Distintos:** Valores Distintos por colunas.<br>
**Quantidade Disponível abaixo do Estoque Mínimo :** Quantidade Disponível abaixo do Estoque Mínimo.<br>

In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("CONTAINER_NAME")

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

In [0]:
food_estoque_lojas_path = (
    f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
    "batch-data/food_estoque_lojas.csv"
)

df_food_estoque_lojas_raw = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(food_estoque_lojas_path)
)


# VISÃO GERAL
from pyspark.sql import functions as F

def overview(df, dataset_name):
    print(f"\n=== VISÃO GERAL: {dataset_name} ===")

    total_records = df.count()
    total_columns = len(df.columns)

    print(f"Total de registros: {total_records}")
    print(f"Total de colunas: {total_columns}")

    print("\nSchema:")
    df.printSchema()

    display(df.limit(5))


# NULOS
def null_analysis(df, dataset_name):
    print(f"\n=== VALORES NULOS: {dataset_name} ===")

    total_records = df.count()

    null_counts = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])

    display(null_counts)

# PRIMARY KEY
def primary_key_analysis(df, primary_key, dataset_name):
    print(f"\n=== CHAVE PRIMÁRIA: {dataset_name} ===")
    print(f"Chave: {primary_key}")

    null_keys = df.filter(
        F.col(primary_key).isNull()
    ).count()

    duplicated_keys = (
        df.groupBy(primary_key)
          .count()
          .filter(F.col("count") > 1)
    )

    total_duplicated_keys = duplicated_keys.count()

    total_records = df.count()
    total_unique = df.dropDuplicates([primary_key]).count()
    total_duplicates = total_records - total_unique

    print(f"Chaves nulas: {null_keys}")
    print(f"Chaves duplicadas: {total_duplicated_keys}")
    print(f"Registros duplicados: {total_duplicates}")

    if total_duplicated_keys > 0:
        print("\nChaves duplicadas:")
        display(duplicated_keys)


# DATAS MÁXIMAS E MÍNIMAS
def date_range_analysis(df, date_column_name, dataset_name):
    print(f"\n=== PERÍODO DOS DADOS: {dataset_name} => COLUNA: {date_column_name} ===")
    
    result = df.select(
        F.min(F.col(date_column_name)).alias("data_minima"),
        F.max(F.col(date_column_name)).alias("data_maxima")
    )
    
    display(result)


# VALORES DISTINTOS
def distinct_analysis(df, dataset_name):
    print(f"\n=== VALORES DISTINTOS: {dataset_name} ===")

    distinct_counts = df.select([
        F.countDistinct(F.col(c)).alias(c)
        for c in df.columns
    ])

    display(distinct_counts)


# QUANTRIDADE DISPONIVE ABAIXO DO ESTOQUE MÍNIMO
def stock_below_minimum_analysis(df, dataset_name):
    result = df.filter(
        F.col("quantidade_disponivel") < F.col("estoque_minimo")
    )

    total_records = result.count()

    print(f"\n=== ESTOQUE ABAIXO DO MÍNIMO: {dataset_name} ===")
    print(f"Total de registros: {total_records}")

    display(result.limit(5))

In [0]:
overview(
    df_food_estoque_lojas_raw,
    "food_estoque_lojas"
)

null_analysis(
    df_food_estoque_lojas_raw,
    "food_estoque_lojas"
)

primary_key_analysis(
    df_food_estoque_lojas_raw,
    "id_loja",
    "food_estoque_lojas"
)

date_range_analysis(
    df_food_estoque_lojas_raw,
    "dt_snapshot",
    "food_estoque_lojas"
)

stock_below_minimum_analysis(
    df_food_estoque_lojas_raw,
    "food_estoque_lojas"
)

distinct_analysis(
    df_food_estoque_lojas_raw,
    "food_estoque_lojas"
)



=== VISÃO GERAL: food_estoque_lojas ===
Total de registros: 736020
Total de colunas: 6

Schema:
root
 |-- id_loja: integer (nullable = true)
 |-- sku: string (nullable = true)
 |-- id_lote: double (nullable = true)
 |-- quantidade_disponivel: integer (nullable = true)
 |-- estoque_minimo: integer (nullable = true)
 |-- dt_snapshot: date (nullable = true)


Primeiros registros:


id_loja,sku,id_lote,quantidade_disponivel,estoque_minimo,dt_snapshot
1,mortadelasalame-sadia-025kg-mortadelabologna,null,75,22,2024-01-01
1,mortadelasalame-sadia-025kg-mortadelabologna,6280.0,6,6,2024-01-15
1,mortadelasalame-sadia-025kg-mortadelabologna,null,37,12,2024-01-29
1,mortadelasalame-sadia-025kg-mortadelabologna,null,158,37,2024-02-12
1,mortadelasalame-sadia-025kg-mortadelabologna,null,2,7,2024-02-26



=== VALORES NULOS: food_estoque_lojas ===


id_loja,sku,id_lote,quantidade_disponivel,estoque_minimo,dt_snapshot
0,0,533509,0,0,0



=== CHAVE PRIMÁRIA: food_estoque_lojas ===
Chave: id_loja
Chaves nulas: 0
Chaves duplicadas: 29
Registros duplicados: 735991

Chaves duplicadas:


...

data_minima,data_maxima
2024-01-01,2026-08-24



=== ESTOQUE ABAIXO DO MÍNIMO: food_estoque_lojas ===
Total de registros: 60316


id_loja,sku,id_lote,quantidade_disponivel,estoque_minimo,dt_snapshot
1,mortadelasalame-sadia-025kg-mortadelabologna,null,2,7,2024-02-26
1,mortadelasalame-sadia-025kg-mortadelabologna,null,1,17,2024-03-11
1,mortadelasalame-sadia-025kg-mortadelabologna,null,22,34,2024-06-17
1,mortadelasalame-sadia-025kg-mortadelabologna,6277.0,1,19,2024-08-26
1,mortadelasalame-sadia-025kg-mortadelabologna,6276.0,9,20,2025-02-10



=== VALORES DISTINTOS: food_estoque_lojas ===


id_loja,sku,id_lote,quantidade_disponivel,estoque_minimo,dt_snapshot
29,1559,6866,1039,96,98
